In [ ]:
# Example 1 — Construct an AppDef (a "local component" is just an AppDef of Roles).
import torchx.specs as specs

resource = specs.Resource(cpu=2, gpu=0, memMB=512)
role = specs.Role(
    name="trainer",
    image="/tmp",                # local_cwd treats a filesystem path as the "image"
    entrypoint="/bin/echo",
    args=["epoch", "0"],
    num_replicas=2,
    resource=resource,
    env={"RANK": "0"},
)
app = specs.AppDef(name="hello-torchx", roles=[role])

assert app.roles[0].num_replicas == 2
assert app.roles[0].resource.cpu == 2
assert app.roles[0].env["RANK"] == "0"
print("AppDef:", app.name, "| roles:", [r.name for r in app.roles],
      "| replicas:", app.roles[0].num_replicas, "| cpu:", app.roles[0].resource.cpu)

In [ ]:
from torchx.specs import get_named_resources
from torchx.schedulers import get_scheduler_factories

p3 = get_named_resources("aws_p3.2xlarge")   # 8 vCPU / 1x V100
g4 = get_named_resources("aws_g4dn.xlarge")  # 4 vCPU / 1x T4
assert (p3.cpu, p3.gpu) == (8, 1), (p3.cpu, p3.gpu, p3.memMB)
assert (g4.cpu, g4.gpu) == (4, 1), (g4.cpu, g4.gpu, g4.memMB)

backends = sorted(get_scheduler_factories().keys())
assert {"local_cwd", "local_docker"} <= set(backends), backends
print("aws_p3.2xlarge:", p3.cpu, "cpu /", p3.gpu, "gpu /", p3.memMB, "MB")
print("scheduler backends:", backends)

In [ ]:
# Example 3 — REAL runtime init: the local_cwd scheduler forks a subprocess per replica.
# dryrun -> run -> wait, then assert the job reached SUCCEEDED (not just an import).
import torchx.specs as specs
from torchx.runner import get_runner
from torchx.specs import AppState

app = specs.AppDef(
    name="smoke",
    roles=[specs.Role(name="greeter", image="/tmp", entrypoint="/bin/echo",
                      args=["hello", "from", "torchx"], num_replicas=1,
                      resource=specs.Resource(cpu=1, gpu=0, memMB=256))],
)
runner = get_runner()
dryrun = runner.dryrun(app, scheduler="local_cwd")
print("dryrun:", type(dryrun).__name__)

handle = runner.run(app, scheduler="local_cwd")
status = runner.wait(handle)
print("app handle:", handle, "| final state:", status.state)
assert status.state == AppState.SUCCEEDED, status.state
runner.close()

In [ ]:
# Example 4 — Invariant/round-trip: the entrypoint + args we define must survive
# materialization into the concrete launch request the scheduler will execute.
import torchx.specs as specs
from torchx.runner import get_runner

app = specs.AppDef(
    name="invariant",
    roles=[specs.Role(name="worker", image="/tmp", entrypoint="/bin/echo",
                      args=["token-42"], num_replicas=1,
                      resource=specs.Resource(cpu=1, gpu=0, memMB=256))],
)
runner = get_runner()
dryrun = runner.dryrun(app, scheduler="local_cwd")
replica = dryrun.request.role_params["worker"][0]   # PopenRequest -> ReplicaParam
assert replica.args[0] == "/bin/echo", replica.args
assert "token-42" in replica.args, replica.args
print("materialized replica args:", replica.args)
runner.close()

In [ ]:
# Example 5 — Known-fragile path: resolve a *named* builtin component ("utils.echo")
# through the component finder + dryrun. The finder validates the component's source
# with torchx.specs.file_linter (an ast-based linter) before materializing the AppDef.
# This is torchx's headline `torchx run utils.echo ...` workflow.
from torchx.runner import get_runner

runner = get_runner()
dryrun = runner.dryrun_component("utils.echo", ["--msg", "hi from torchx"],
                                 scheduler="local_cwd")
assert dryrun.request is not None
print("component 'utils.echo' dryrun:", type(dryrun).__name__)
print(dryrun)
runner.close()